# Phase 3 — Retrieval experiments: chunk config × embedder × query language × reranker

Builds the FAISS indices and runs the full retrieval grid: 3 chunk configs
(`baseline_buggy`, `400_80`, `250_50`) × 2 embedders (`bge-small-en-v1.5` free,
`text-embedding-3-large` paid, winning config only) × 3 query-language variants for
the exam-dev set (raw Hebrew, Claude-translated English, translated + glossary) ×
reranker on/off — evaluated against **two independent gold-label sources**: the
synthetic set's exact gold chunk, and expert exam references (via
`eval/parse_exam.py`'s `resolve_reference`, imported lazily so this notebook never
needs pymupdf). See `derma_guide_plan.md` Phase 3 for the full results this should
reproduce, including the chunk-size ground-truth-geometry artifact (denser chunking
mechanically inflates synthetic Hit@k; wider chunks mechanically inflate exam Hit@k) —
read the numbers this notebook produces with that caveat in mind, not at face value.

CPU is fine — no GPU needed for `bge` (a ~130MB model) or the OpenAI API calls, but
embedding all three chunk configs' corpora (2,521 + 6,115 + 9,770 chunks) is slow on a
weak CPU (this dev sandbox: several hours); Colab's free CPU runtime is meaningfully
faster and is the intended way to run this notebook.

**Files to upload** into `/content/drive/MyDrive/derma_force_eval/`:
```
pipeline/data/chunks/
  chunks.jsonl  chunks_250_50.jsonl  chunks_baseline_buggy.jsonl
  chunks_baseline_buggy_embedded.jsonl        <- pre-embedded OpenAI vectors, reused not re-billed
eval/
  build_index.py  eval_retrieval.py  parse_exam.py  audit_exams.py  translate_queries.py
  data/
    exam_parsed/all.jsonl                      <- from notebooks/01_parse_exams.ipynb
    exam_query_translations.json               <- already-collected Batch API translations
                                                   (skip step 5 below if uploading this)
backend/.env                                    <- only if step 5 needs to (re)run translate_queries.py
```
The layout under `/content/derma_force/` mirrors the repo root exactly.

## 1. Setup

In [ ]:
!pip install -q faiss-cpu sentence-transformers openai anthropic python-dotenv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

EVAL_DRIVE_DIR = '/content/drive/MyDrive/derma_force_eval'

import os
REPO_ROOT = '/content/derma_force'
if not os.path.exists(REPO_ROOT):
    os.symlink(EVAL_DRIVE_DIR, REPO_ROOT)
print("repo root:", REPO_ROOT, "->", os.readlink(REPO_ROOT) if os.path.islink(REPO_ROOT) else "(real dir)")
!ls {REPO_ROOT}/pipeline/data/chunks
!ls {REPO_ROOT}/eval/data

## 2. Build the FAISS indices

`bge` across the full chunk-config grid; `openai` for `400_80` only (the config
carried forward — see Phase 3's discussion of why it's the honest choice, not the
best-scoring one) plus `baseline_buggy` (reusing its already-embedded OpenAI vectors,
real product cost already spent, rather than re-billing).

In [ ]:
%cd {REPO_ROOT}/eval
for config in ["baseline_buggy", "400_80", "250_50"]:
    !python3 build_index.py --config {config} --embedder bge
!python3 build_index.py --config 400_80 --embedder openai
!python3 build_index.py --config baseline_buggy --embedder openai

## 3. Retrieval eval — all configs, bge

Hit@1/3/5/10 + MRR@10 against synthetic gold and (for `400_80`, once translations
exist) all three exam-dev query-language variants.

In [ ]:
!python3 eval_retrieval.py --configs baseline_buggy 400_80 250_50 --embedder bge

## 4. Retrieval eval — openai, winning config + baseline for comparison

In [ ]:
!python3 eval_retrieval.py --configs 400_80 baseline_buggy --embedder openai

## 5. (Optional) Translate the exam-dev queries — only if `exam_query_translations.json`
wasn't uploaded already

Batch API, async (submit now, `status` until `ended_at` is set, then `collect`) — see
`derma_guide_plan.md` Phase 3 for real timing (this ran ~1h50m at 1,220 requests
in the equivalent synthetic-QA job; expect a similar order of magnitude here at
~500 requests). **Skip this cell entirely if `eval/data/exam_query_translations.json`
is already present** — steps 3/4 above already used it if so.

In [ ]:
!python3 translate_queries.py submit
# then, after some time:
# !python3 translate_queries.py status
# and once ended_at is set:
# !python3 translate_queries.py collect
# !python3 eval_retrieval.py --configs 400_80 --embedder bge   # re-run to pick up the new variants

## 6. Reranker pass — winning config only

`cross-encoder/ms-marco-MiniLM-L-6-v2`, retrieve top-20 then rerank to top-10. Slow on
CPU (a few minutes per query set × variant) — this is the one step where Colab's
CPU is materially faster than a constrained local sandbox.

In [ ]:
!pip install -q sentence-transformers  # already installed above; cross-encoder needs it too
!python3 eval_retrieval.py --configs 400_80 --embedder bge --rerank
!python3 eval_retrieval.py --configs 400_80 --embedder openai --rerank

## 7. Pull the merged results back to Drive

In [ ]:
import shutil, os
os.makedirs(f"{EVAL_DRIVE_DIR}/eval/data/indices", exist_ok=True)
shutil.copy(f"{REPO_ROOT}/eval/data/retrieval_results.json", f"{EVAL_DRIVE_DIR}/eval/data/retrieval_results.json")
for f in os.listdir(f"{REPO_ROOT}/eval/data/indices"):
    shutil.copy(f"{REPO_ROOT}/eval/data/indices/{f}", f"{EVAL_DRIVE_DIR}/eval/data/indices/{f}")
if os.path.exists(f"{REPO_ROOT}/eval/data/exam_query_translations.json"):
    shutil.copy(f"{REPO_ROOT}/eval/data/exam_query_translations.json",
                f"{EVAL_DRIVE_DIR}/eval/data/exam_query_translations.json")
print("saved -> retrieval_results.json + indices/ + exam_query_translations.json")

## Next steps (Phase 4/5)

With `retrieval_results.json` and the built indices on Drive:
- Render the plots (`outputs/make_plots.py`, or copy the numbers into the report).
- QLoRA fine-tune (`notebooks/03_finetune.ipynb`) uses the same `400_80` chunk
  config's retrieval to build its training set (`training/prepare_dataset.py`).
- The 4-arm generation eval (`notebooks/04_generation_eval.ipynb`) reuses the
  `400_80__bge` index built here directly.